# Nail U-Net Training for Google Colab

This notebook trains a binary fingernail segmentation U-Net using the split dataset layout:

- `train/images`, `train/masks`
- `val/images`, `val/masks`
- `test/images`, `test/masks`

It saves a PyTorch `state_dict` that can be loaded directly by the Raspberry Pi inference code in `vision.py`.


## 1. Runtime Setup

In Colab, switch to `Runtime -> Change runtime type -> GPU` before training.


In [ ]:
!pip install -q torch torchvision matplotlib pillow tqdm

In [ ]:
# Kaggle datasets are available under /kaggle/input/

## 2. Paths and Hyperparameters

Update `DATASET_ROOT` to where your dataset lives in Colab or Google Drive.


In [ ]:
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/nail-segmentation-dataset-v2/NailSegmentationDatasetV2')
OUTPUT_DIR = Path('/kaggle/working/nail_unet_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (256, 256)
BATCH_SIZE = 16
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
THRESHOLD = 0.5
SEED = 42

BEST_MODEL_PATH = OUTPUT_DIR / 'nail_unet_best.pt'
LAST_MODEL_PATH = OUTPUT_DIR / 'nail_unet_last.pt'

for split in ['train', 'val', 'test']:
    print(split, (DATASET_ROOT / split / 'images').exists(), (DATASET_ROOT / split / 'masks').exists())

In [ ]:
import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 3. Dataset and Dataloaders

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}


class NailSegmentationDataset(Dataset):
    def __init__(self, root: Path, split: str, image_size=(512, 512), augment=False):
        self.root = Path(root)
        self.split = split
        self.images_dir = self.root / split / 'images'
        self.masks_dir = self.root / split / 'masks'
        self.image_size = image_size
        self.augment = augment

        self.samples = []
        for image_path in sorted(self.images_dir.iterdir()):
            if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            mask_path = self.masks_dir / f'{image_path.stem}.png'
            if mask_path.exists():
                self.samples.append((image_path, mask_path))

        if not self.samples:
            raise RuntimeError(f'No image-mask pairs found in {self.images_dir} and {self.masks_dir}')

        self.image_transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
        ])
        self.mask_resize = transforms.Resize(image_size, interpolation=transforms.InterpolationMode.NEAREST)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, mask_path = self.samples[index]
        image = Image.open(image_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')

        if self.augment and random.random() < 0.5:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
        if self.augment and random.random() < 0.5:
            image = image.transpose(Image.FLIP_TOP_BOTTOM)
            mask = mask.transpose(Image.FLIP_TOP_BOTTOM)

        image_tensor = self.image_transform(image)
        mask = self.mask_resize(mask)
        mask_tensor = transforms.ToTensor()(mask)
        mask_tensor = (mask_tensor > 0.5).float()

        return image_tensor, mask_tensor, image_path.name


train_dataset = NailSegmentationDataset(DATASET_ROOT, 'train', image_size=IMAGE_SIZE, augment=True)
val_dataset = NailSegmentationDataset(DATASET_ROOT, 'val', image_size=IMAGE_SIZE, augment=False)
test_dataset = NailSegmentationDataset(DATASET_ROOT, 'test', image_size=IMAGE_SIZE, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print('Train:', len(train_dataset))
print('Val:', len(val_dataset))
print('Test:', len(test_dataset))

In [ ]:
import matplotlib.pyplot as plt

images, masks, names = next(iter(train_loader))
fig, axes = plt.subplots(3, 2, figsize=(8, 12))
for row in range(3):
    axes[row, 0].imshow(images[row].permute(1, 2, 0).numpy())
    axes[row, 0].set_title(names[row])
    axes[row, 0].axis('off')
    axes[row, 1].imshow(masks[row, 0].numpy(), cmap='gray')
    axes[row, 1].set_title('mask')
    axes[row, 1].axis('off')
plt.tight_layout()

## 4. U-Net Model

This matches the structure used in `vision.py` so the saved weights load cleanly on the Raspberry Pi.


In [ ]:
import torch.nn as nn


class TwoConvLayers(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(out_channels),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        return self.model(x)


class Encoder(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = TwoConvLayers(in_channels, out_channels)
        self.max_pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        skip = self.block(x)
        return self.max_pool(skip), skip


class Decoder(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.transpose = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.block = TwoConvLayers(in_channels, out_channels)

    def forward(self, x, skip):
        x = self.transpose(x)
        x = torch.cat([x, skip], dim=1)
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels: int = 3, num_classes: int = 1):
        super().__init__()
        self.enc_block1 = Encoder(in_channels, 32)
        self.enc_block2 = Encoder(32, 64)
        self.enc_block3 = Encoder(64, 128)
        self.enc_block4 = Encoder(128, 256)
        self.bottleneck = TwoConvLayers(256, 512)
        self.dec_block1 = Decoder(64, 32)
        self.dec_block2 = Decoder(64, 32)
        self.dec_block3 = Decoder(64, 32)
        self.dec_block4 = Decoder(64, 32)
        self.output = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        x, skip1 = self.enc_block1(x)
        x, skip2 = self.enc_block2(x)
        x, skip3 = self.enc_block3(x)
        x, skip4 = self.enc_block4(x)
        x = self.bottleneck(x)
        x = self.dec_block1(x, skip4)
        x = self.dec_block2(x, skip3)
        x = self.dec_block3(x, skip2)
        x = self.dec_block4(x, skip1)
        return self.output(x)

## 5. Loss, Metrics, and Training Helpers

In [ ]:
from tqdm.auto import tqdm


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(probs.size(0), -1)
        targets = targets.view(targets.size(0), -1)
        intersection = (probs * targets).sum(dim=1)
        union = probs.sum(dim=1) + targets.sum(dim=1)
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()


def dice_score_from_logits(logits, targets, threshold=0.5, smooth=1.0):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1)
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return dice.mean().item()


bce_loss = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([10.0]).to(device))
dice_loss = DiceLoss()


def segmentation_loss(logits, targets):
    return 0.1 * bce_loss(logits, targets) + 0.9 * dice_loss(logits, targets)


def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    total_dice = 0.0

    autocast_enabled = device.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=autocast_enabled and is_train)

    progress = tqdm(loader, leave=False)
    for images, masks, _ in progress:
        images = images.to(device)
        masks = masks.to(device)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=autocast_enabled):
                logits = model(images)
                loss = segmentation_loss(logits, masks)

            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        batch_dice = dice_score_from_logits(logits.detach(), masks, threshold=THRESHOLD)
        total_loss += loss.item() * images.size(0)
        total_dice += batch_dice * images.size(0)
        progress.set_postfix(loss=f'{loss.item():.4f}', dice=f'{batch_dice:.4f}')

    dataset_size = len(loader.dataset)
    return total_loss / dataset_size, total_dice / dataset_size

## 6. Train the Model

In [ ]:
model = UNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

history = []
best_val_dice = -1.0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_dice = run_epoch(model, train_loader, optimizer=optimizer)
    val_loss, val_dice = run_epoch(model, val_loader, optimizer=None)
    scheduler.step(val_dice)

    record = {
        'epoch': epoch,
        'train_loss': train_loss,
        'train_dice': train_dice,
        'val_loss': val_loss,
        'val_dice': val_dice,
        'lr': optimizer.param_groups[0]['lr'],
    }
    history.append(record)
    print(record)

    checkpoint = {
        'model_state_dict': model.state_dict(),
        'history': history,
        'image_size': IMAGE_SIZE,
        'threshold': THRESHOLD,
    }
    torch.save(checkpoint, LAST_MODEL_PATH)

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(checkpoint, BEST_MODEL_PATH)
        print(f'Saved new best model to {BEST_MODEL_PATH}')

print('Best validation Dice:', best_val_dice)

In [ ]:
import pandas as pd

history_df = pd.DataFrame(history)
history_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['train_dice'], label='train')
axes[1].plot(history_df['epoch'], history_df['val_dice'], label='val')
axes[1].set_title('Dice')
axes[1].legend()
plt.tight_layout()

## 7. Visualize Predictions on the Test Set

In [ ]:
best_checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

test_loss, test_dice = run_epoch(model, test_loader, optimizer=None)
print({'test_loss': test_loss, 'test_dice': test_dice})

In [ ]:
images, masks, names = next(iter(test_loader))
images = images.to(device)
with torch.no_grad():
    logits = model(images)
    preds = (torch.sigmoid(logits) > THRESHOLD).float().cpu()

images = images.cpu()
fig, axes = plt.subplots(4, 3, figsize=(10, 14))
for row in range(4):
    axes[row, 0].imshow(images[row].permute(1, 2, 0).numpy())
    axes[row, 0].set_title(names[row])
    axes[row, 0].axis('off')

    axes[row, 1].imshow(masks[row, 0].numpy(), cmap='gray')
    axes[row, 1].set_title('ground truth')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(preds[row, 0].numpy(), cmap='gray')
    axes[row, 2].set_title('prediction')
    axes[row, 2].axis('off')

plt.tight_layout()

## 8. Export a Raspberry Pi Ready Model File

The Pi inference code can load either the checkpoint or a plain `state_dict`. This cell writes the plain version.


In [ ]:
PI_MODEL_PATH = OUTPUT_DIR / 'nail_unet_pi_state_dict.pt'
torch.save(model.state_dict(), PI_MODEL_PATH)
print('Saved Pi-ready weights to', PI_MODEL_PATH)

## 9. What to Copy Back Into the Project

Copy `nail_unet_pi_state_dict.pt` into your repo as:

```text
models/nail_unet.pt
```

That file can then be loaded by the runtime segmentation code on the Raspberry Pi.
